In [ ]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
df.head()


(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df["TotalCharges"].isna().sum())

11


In [ ]:
df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
print(df.duplicated().sum())
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True) * 100)

0
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [ ]:
# Churn as 1/0 so we can average it to get churn rate
df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})

# SeniorCitizen as Yes/No instead of 1/0 (clearer labels in charts)
df["SeniorCitizen"] = df["SeniorCitizen"].map({1: "Yes", 0: "No"})

# Group tenure into bands
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
)

In [ ]:
df.to_csv("telco_clean.csv", index=False)

# **Model Training**

In [ ]:
from sklearn.model_selection import train_test_split

# Features = everything except ID, the answer columns, and the helper column
X = df.drop(columns=["customerID", "Churn", "ChurnFlag", "TenureGroup"])
y = df["ChurnFlag"]

# Convert text columns (Yes/No, Fiber optic, etc.) into numbers
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(5634, 30) (1409, 30)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

log_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, log_pred))
print("ROC-AUC:", roc_auc_score(y_test, log_prob))
print(classification_report(y_test, log_pred))

Accuracy: 0.7402413058907026
ROC-AUC: 0.8414012245214291
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", min_samples_leaf=5)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

Accuracy: 0.772888573456352
ROC-AUC: 0.8442713580820999


In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(10))

tenure                             0.182935
TotalCharges                       0.137688
MonthlyCharges                     0.102342
Contract_Two year                  0.100447
InternetService_Fiber optic        0.066417
PaymentMethod_Electronic check     0.056789
Contract_One year                  0.042678
OnlineSecurity_Yes                 0.034752
TechSupport_Yes                    0.024707
TechSupport_No internet service    0.020417
dtype: float64


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_prob = cross_val_predict(
    RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", min_samples_leaf=5),
    X, y, cv=cv, method="predict_proba"
)[:, 1]

print(all_prob[:5])


[0.82590137 0.07558864 0.64861467 0.09815253 0.78355191]


In [ ]:
df["ChurnProbability"] = all_prob.round(3)

df["RiskLevel"] = pd.cut(
    df["ChurnProbability"],
    bins=[-0.01, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"]
)

print(df["RiskLevel"].value_counts())

RiskLevel
Low       4065
Medium    1712
High      1266
Name: count, dtype: int64


In [ ]:
print(df.groupby("RiskLevel", observed=True)["ChurnFlag"].mean())

RiskLevel
Low       0.085855
Medium    0.376752
High      0.691153
Name: ChurnFlag, dtype: float64


In [ ]:
df.to_csv("telco_final.csv", index=False)